### **Extraer el `texto`, `autor` y `tags` y exportarlos a un archivo `JSON`**

Vamos a trabajar en el enlace:

https://quotes.toscrape.com/

#### **Instalación Entorno virtual y Scrapy**

1. Ubicandonos en el directorio (ruta) en el que deseemos instalar el entorno virtual, ya desde la consola de VSCode o cmd o Powershell, escribimos el siguiente comando:

In [ ]:
python -m venv .\venv

En mi caso yo lo hice en la ruta: `C:\github\proyectos` y utilizando `VSCode`:

<center><img src="https://i.postimg.cc/Hn0cgNwV/ws-364.png"></center>
<center><img src="https://i.postimg.cc/ncZ9kHx7/ws-365.png"></center>

Luego activamos el entorno virtual con la siguiente ruta (desde `cmd`):

In [ ]:
venv\Scripts\activate

<center><img src="https://i.postimg.cc/BQgL7dZz/ws-366.png"></center>

3. Instalamos Scrapy dentro del entorno:

<center><img src="https://i.postimg.cc/vZ21bfVL/ws-367.png"></center>

4. Ingresamos al entorno utilizando la ruta `venv\Scripts\activate`. Luego, revisamos si la instalación de Scrapy se hizo correctamente y vemos su versión:

<center><img src="https://i.postimg.cc/W11DBXh1/ws-368.png"></center>

Para salir del entorno virtual utilizamos el comando **`deactivate`**

#### **Crear proyecto Spider**


Vamos a crear un nuevo proyecto al cual le daremos el nombre de `scrapy_projects`. Debemos escribir el siguiente comando:

<center><img src="https://i.postimg.cc/Mp7r3C9g/ws-369.png"></center>
<center><img src="https://i.postimg.cc/KYmp60Vj/ws-370.png"></center>

Podemos comenzar NUESTRO PRIMER SPIDER de la siguiente manera:

1. cd scrapy_projects
2. Vamos a utilizar el siguiente enlace: https://quotes.toscrape.com/
3. scrapy gendspider `<nombre_de_nuestra_spider> <enlace(no es necesario el https://)>`

<center><img src="https://i.postimg.cc/V6V7bmN0/ws-371.png"></center>

Luego, se creará de manera automática el siguiente archivo .py:

<center><img src="https://i.postimg.cc/vBBqBcGd/ws-372.png"></center>

#### **Paso a paso**

Vamos a inspeccionar el sitio:

<center><img src="https://i.postimg.cc/NGP2bPqQ/ws-358.png"></center>

Localizar cada `articulo` de cita:

<center><img src="https://i.postimg.cc/YqGGPfMY/ws-359.png"></center>

Localizar el `texto` de la cita:

<center><img src="https://i.postimg.cc/vZ51YY1m/ws-360.png"></center>

Localizar el `autor` de la cita:

<center><img src="https://i.postimg.cc/vmM4DJ9t/ws-361.png"></center>

Localizar los `tags` de la cita:

<center><img src="https://i.postimg.cc/tCKZGZ4f/ws-362.png"></center>

Localizar si existe un botón de `Siguiente` para pasar a la siguiente página a continuar el proceso:

<center><img src="https://i.postimg.cc/zfjVhxFZ/ws-363.png"></center>

#### **Código**

Por tanto, escribimos el siguiente código:

In [ ]:
import scrapy


class QuotesToScrapeSpider(scrapy.Spider):
    name = "quotes_to_scrape"
    allowed_domains = ["quotes.toscrape.com"]
    start_urls = ["https://quotes.toscrape.com"]

    def parse(self, response):
        # Cada vez que queramos localizar un elemento, vamos a utilizar la variable 'response'
        for quote in response.xpath('//div[@class="quote"]'):
            yield {
                # El método 'extract_first() aún se utiliza, pero ahora se recomienda usar get() ---> Extrae 1 elemento
                'text': quote.xpath('./span[@class="text"]/text()').extract_first(),
                'author': quote.xpath('.//small[@class="author"]/text()').extract_first(),
                # El método 'extract() aún se utiliza, pero ahora se recomienda usar getall() ---> Extrae varios elementos --> Lista
                'tags': quote.xpath('.//div[@class="tags"]/a[@class="tag"]/text()').extract()
            }

        # El método 'extract_first() aún se utiliza, pero ahora se recomienda usar get() ---> Extrae 1 elemento
        next_page = response.xpath('//li[@class="next"]/a/@href').extract_first()
        if next_page is not None:
            # Lo mismo --> yield scrapy.Request(response.urljoin(next_page))
            yield response.follow(next_page, self.parse)


Este codigo debemos escribirlo en el archivo `quotes_to_scrape.py`:

<center><img src="https://i.postimg.cc/1X3mLTwQ/ws-373.png"></center>

#### **¿Qué acaba de pasar?**

Al ejecutar el comando scrapy runspider quotes_spider.py, Scrapy buscó una definición Spider en su interior y la ejecutó a través de su motor de rastreo (crawler engine).

El rastreo (crawl) comenzó haciendo peticiones (requests) a las URLs definidas en el atributo start_urls (en este caso, sólo la URL para las citas) y llamó al método callback por defecto parse, pasando el objeto response como argumento. En el callback parse, hacemos un bucle a través de los elementos de la cita utilizando un selector XPATH, producimos un diccionario Python con el texto de la cita extraído, el autor y los tags, buscamos un enlace a la página siguiente y programamos otra solicitud (request) utilizando el mismo método parse como callback.

Aquí se nota una de las principales ventajas de Scrapy: las peticiones se programan y procesan de forma asíncrona. Esto significa que Scrapy no necesita esperar a que una petición termine y sea procesada, puede enviar otra petición o hacer otras cosas mientras tanto. Esto también significa que otras peticiones pueden seguir adelante incluso si alguna petición falla o se produce un error mientras se procesa.

Mientras que esto le permite hacer rastreos muy rápidos (enviando múltiples peticiones concurrentes al mismo tiempo, de una manera tolerante a fallos) Scrapy también le da el control sobre la cortesía del rastreo a través de algunos ajustes. Puedes hacer cosas como establecer un tiempo de espera entre cada petición, limitar la cantidad de peticiones concurrentes por dominio o por IP, e incluso usar una extensión de auto-throttling que intente calcular esto automáticamente.

#### **Ejecución**

Para ejecutar el bot tenemos que estar ubicados en la carpeta que tiene el archivo "`scrapy.cfg`". El nombre de esta carpeta es el mismo que el nombre del proyecto que creamos. En este caso, el nombre de mi proyecto es "`scrapy_projects`". Así que ahora que nos ubicamos en esa carpeta, dentro de la terminal, vamos a escribir el comando:

In [ ]:
scrapy crawl quotes_to_scrape

<center><img src="https://i.postimg.cc/Y2NttQxP/ws-374.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/4xyGvqTC/ws-375.png"></center>

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
scrapy crawl quotes_to_scrape -o quotes_to_scrape.json

<center><img src="https://i.postimg.cc/Fs9Nz1WH/ws-376.png"></center>